# Get To Know A Dataset: ConfUSIus Functional Ultrasound Imaging Dataset Collection

This notebook is a draft AWS Open Data Sponsorship Program tutorial for the ConfUSIus Functional Ultrasound Imaging Dataset Collection. It introduces the planned AWS-hosted collection, its organization, data formats, and example access patterns.

The final bucket name will be filled in after AWS confirms that the dedicated AWS account has joined the Open Data billing organization and the S3 bucket has been created with the AWS-supplied CloudFormation template.


## Dataset overview

ConfUSIus is a Python toolkit for functional ultrasound imaging (fUSI) analysis. The ConfUSIus Dataset Collection brings together openly available fUSI datasets and derived templates from multiple studies and laboratories. Where possible, datasets are organized or re-exported with BIDS/fUSI-BIDS conventions so users can browse, download, and load data with consistent tooling.

The collection includes fUSI-BIDS datasets such as Nunez-Elizalde 2022, Pereira 2025, Cybis Pereira 2026, Pepe Mariani 2026, Landemard 2026, and Khallaf 2026, plus ConfUSIus-compatible fUSI templates. BrainGlobe atlases are not part of this collection; they are fetched separately through BrainGlobe.


### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The planned S3 layout keeps each source dataset independently addressable under a stable ConfUSIus identifier:

```text
s3://<bucket>/
  README.md
  datasets/
    nunez-elizalde-2022-bids/
    pereira-2025-bids/
    cybis-pereira-2026-bids/
    pepe-mariani-2026-bids/
    landemard-2026-bids/
    khallaf-2026-bids/
  templates/
    huang-2025-template/
    pepe-mariani-2026-template/
  metadata/
    datasets.json
    checksums/
    manifests/
  docs/
```

The `datasets/` entries are BIDS or fUSI-BIDS dataset roots. The `templates/` entries contain small derived NIfTI templates. The `metadata/` prefix is reserved for machine-readable collection metadata, manifests, and checksums.


In [ ]:
# This notebook requires the following additional libraries:
# boto3
# botocore
# confusius
# matplotlib

from pathlib import Path

import boto3
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "TODO_AFTER_AWS_ACCOUNT_LINKING"
REGION = "us-west-2"

s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))


In [ ]:
def list_prefix(prefix: str = "", max_keys: int = 20) -> list[str]:
    """List keys or common prefixes in the public bucket."""
    if BUCKET.startswith("TODO"):
        print("S3 bucket placeholder: fill BUCKET after AWS account linking and bucket creation.")
        return []

    response = s3.list_objects_v2(
        Bucket=BUCKET,
        Prefix=prefix,
        Delimiter="/",
        MaxKeys=max_keys,
    )
    prefixes = [item["Prefix"] for item in response.get("CommonPrefixes", [])]
    keys = [item["Key"] for item in response.get("Contents", [])]
    return prefixes + keys

list_prefix()


### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

The main imaging format is NIfTI (`.nii` / `.nii.gz`) with JSON sidecars and TSV tables following BIDS/fUSI-BIDS conventions where possible. Some datasets include derivative maps, tabular behavioural or event files, videos, or source files. The collection intentionally preserves source-dataset differences rather than pretending all studies have identical structure.

Recommended access pattern:

1. Start from `metadata/datasets.json` and the ConfUSIus dataset documentation to choose a dataset.
2. Browse the corresponding `datasets/<dataset-id>/` prefix.
3. Download the subset needed for analysis.
4. Load NIfTI files with `confusius.load`, which returns VoxelData arrays backed by xarray.

License terms vary by source dataset. Check the dataset documentation before reuse or redistribution.


### Q: Can you show us an example of downloading and loading data from your dataset?

The final notebook will demonstrate downloading a small Nunez-Elizalde 2022 subset from S3, then loading one power Doppler NIfTI file with ConfUSIus. ConfUSIus is the recommended Python entry point because it reads BIDS/fUSI-BIDS sidecars and returns spatially referenced xarray `VoxelData` arrays.

Until the AWS bucket exists, the code below documents the intended S3 access pattern and exits early when the bucket placeholder is still present.


In [ ]:
import confusius as cf

example_key = (
    "datasets/nunez-elizalde-2022-bids/"
    "sub-CR020/ses-20191122/fusi/"
    "sub-CR020_ses-20191122_task-spontaneous_acq-slice03_pwd.nii.gz"
)
local_path = Path("data") / Path(example_key).name

if BUCKET.startswith("TODO"):
    print("Draft mode: would download", f"s3://{BUCKET}/{example_key}")
    print("After bucket creation, this file will be loaded with: pwd = cf.load(local_path)")
else:
    local_path.parent.mkdir(exist_ok=True)
    s3.download_file(BUCKET, example_key, str(local_path))

    pwd = cf.load(local_path)
    print(pwd)
    print("VoxelData dimensions:", pwd.dims)
    print("Spatial coordinates:", [coord for coord in ("z", "y", "x") if coord in pwd.coords])


ConfUSIus keeps the data as an xarray object, so standard labelled-array operations work directly. For example, `pwd.mean("time")` computes a mean power Doppler image while preserving the spatial coordinates needed for plotting, registration, and atlas operations.


### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

A simple first visual for fUSI data is the mean power Doppler image over time. In the final notebook, this section will plot the mean image from the small Nunez-Elizalde example above.


In [ ]:
import matplotlib.pyplot as plt

if "pwd" not in globals():
    print("Draft mode: run the S3 download cell after the bucket is available to create this plot.")
else:
    mean_pwd = pwd.mean("time").squeeze()
    plt.figure(figsize=(5, 4))
    plt.imshow(mean_pwd, cmap="magma")
    plt.title("Mean power Doppler signal")
    plt.axis("off")
    plt.colorbar(label="Power Doppler")


### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

ConfUSIus examples show analyses such as registration across sessions, atlas-based functional connectivity, GLM activation mapping, and searchlight decoding of behavioural variables. A concrete example is the Cybis Pereira 2026 speed dataset: ConfUSIus can load a subset of the fUSI-BIDS tree, align recordings, and fit models relating vascular activity to running speed.

For the initial AWS draft, the reproducible code for this section will be kept small. The final notebook should point to the full ConfUSIus examples for larger analyses and include a lightweight summary computed from a small S3 subset.


### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

Community challenge: build cross-dataset fUSI analysis methods that are robust to differences in species, acquisition geometry, sampling rate, preprocessing, and experimental design. For example, can a common representation of fUSI vascular dynamics learned on one dataset transfer to another dataset or task?

Advice for users:

- Treat each source dataset as an independent study with its own license, acquisition protocol, and metadata quality.
- Start with one small subset and validate all spatial/time axes before scaling up.
- Use BIDS/fUSI-BIDS metadata when present, but inspect dataset-specific documentation and sidecars.
- Keep analysis outputs separate from source data and record exact S3 keys, ConfUSIus version, and citations.


## More examples

Published ConfUSIus examples using collection datasets include:

- Registration of two sessions from the same subject: https://confusius.tools/latest/examples/_built/registration/register_volume_same_subject/
- First-level GLM analysis of fUSI data: https://confusius.tools/latest/examples/_built/glm/first_level/
- Lagged first-level GLM with a continuous regressor: https://confusius.tools/latest/examples/_built/glm/first_level_continuous/
- Searchlight decoding of a continuous variable: https://confusius.tools/latest/examples/_built/decoding/searchlight_speed/
- Atlas-based functional connectivity examples: https://confusius.tools/latest/examples/


## Draft status

Before final submission, replace the bucket placeholder, run the notebook against the public S3 bucket, verify the example keys, and clear all outputs before committing.
